# Vexil private binary QA: fullpit

In [ ]:

import os,sys,json,hashlib,tarfile
from pathlib import Path
payload_path=Path('/kaggle/input/datasets/sangtruong/vexil-redteam-train291-binary-20261113/qa_payload.bin')
print('DATASET_PRESENT',payload_path.is_file())
assert payload_path.is_file(), str(payload_path)
assert hashlib.sha256(payload_path.read_bytes()).hexdigest()=='8d9c34c7a4547b6bc29587ffafb4ce34a939b7df90548328b92d5407760c3e8d'
dest=Path('/kaggle/working')
with tarfile.open(payload_path,'r:gz') as tf:
    assert all(m.name.startswith('qa-project/') and '..' not in m.name for m in tf)
    tf.extractall(path=dest,filter='data')
project=dest/'qa-project'
sys.path.insert(0,str(project))
os.chdir(project)
import pandas as pd
manifest=pd.read_csv(project/'cache/vexil_takehome_outputs/match_manifest.csv')
train=manifest.loc[manifest.partition=='train']
assert len(train)==291
for mid in train.match_id:
    key=hashlib.sha256(json.dumps(['/matches/'+str(int(mid)),{}],sort_keys=True).encode()).hexdigest()
    assert (project/'cache/api-cache'/(key+'.json.gz')).is_file(),('missing fixture',int(mid))
import research.robustness_study as rs
rs.CACHE=project/'cache'
rs.OUT=dest/'qa-results'
print('INPUT_TRAIN_ONLY_VERIFIED',len(train))

from tests.test_notebook_contracts import load_definitions
import math
ns=load_definitions()
api=ns['OpenDota'](cache_dir=rs.CACHE/'api-cache')
cases=0; games=0
for mid in train.match_id:
    g=api.get('/matches/'+str(int(mid)))
    games+=1
    for minute in range(int(g['duration'])//60+1):
        original=ns['snapshot_features'](g,minute)
        truncated=ns['snapshot_features'](ns['prefix_truncated'](g,minute),minute)
        ns['assert_same_features'](original,truncated)
        raw=ns['item_purchase_snapshot'](g,minute)
        cut=ns['item_purchase_snapshot'](ns['prefix_truncated'](g,minute),minute)
        assert all((math.isnan(x) and math.isnan(y)) or x==y for x,y in zip(raw,cut))
        cases+=1
assert api.calls==0 and games==291 and cases==10683
result=dict(status='PASS',actual_cases=cases,games=games,real_api_requests=api.calls,
  validation_103_scored=0,heldout_106_scored=0,source='Kaggle 291 cached training OpenDota games')
(rs.OUT/'full_train_10683_pit_receipt.json').parent.mkdir(parents=True,exist_ok=True)
(rs.OUT/'full_train_10683_pit_receipt.json').write_text(json.dumps(result,sort_keys=True,indent=2)+'\n')
print('KAGGLE_FULL_TRAIN_PIT_PASS',json.dumps(result,sort_keys=True))
